In [ ]:
!pip install groq pandas

Load API Key

In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()

Initialize Client

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

Test API

In [ ]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say hello in one sentence"}]
)

print(response.choices[0].message.content)

Hello, how can I assist you today?


Create Function

In [ ]:
import time

def call_groq(prompt):
    start = time.time()

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    end = time.time()

    return {
        "model": "Groq",
        "response": response.choices[0].message.content,
        "latency": round(end - start, 2),
        "tokens": response.usage.total_tokens
    }

Cost Estimation

In [ ]:
def estimate_cost(tokens):
    price_per_token = 0.000001  # approx
    return round(tokens * price_per_token, 6)

Quality Scoring

In [ ]:
def score_quality(response, criteria):
    score = 0

    if "accuracy" in criteria and len(response) > 30:
        score += 3

    if "brevity" in criteria and len(response.split()) < 100:
        score += 3

    if "tone" in criteria:
        if any(word in response.lower() for word in ["thank", "please", "happy"]):
            score += 4

    return score

Run Benchmark

In [ ]:
import pandas as pd

prompt = "Explain refund policy for an e-commerce website"
criteria = ["accuracy", "brevity", "tone"]

result = call_groq(prompt)

result["cost"] = estimate_cost(result["tokens"])
result["quality"] = score_quality(result["response"], criteria)

df = pd.DataFrame([result])

df

,model,response,latency,tokens,cost,quality
0,Groq,A refund policy is an essential component of a...,1.86,766,0.000766,7


Clean Output

In [ ]:
df[["model", "latency", "tokens", "cost", "quality"]]

,model,latency,tokens,cost,quality
0,Groq,1.86,766,0.000766,7


In [ ]:
Multiple Prompts

In [ ]:
prompts = [
    "Explain refund policy",
    "Write a professional email",
    "What is AI?"
]

results = []

for p in prompts:
    res = call_groq(p)
    res["cost"] = estimate_cost(res["tokens"])
    res["quality"] = score_quality(res["response"], ["accuracy", "brevity", "tone"])
    res["prompt"] = p
    results.append(res)

df = pd.DataFrame(results)

df

,model,response,latency,tokens,cost,quality,prompt
0,Groq,A refund policy is a set of rules and guidelin...,1.27,571,0.000571,3,Explain refund policy
1,Groq,Subject: Proposal for Collaboration on Project...,0.88,397,0.000397,7,Write a professional email
2,Groq,Artificial Intelligence (AI) refers to the sim...,1.18,664,0.000664,3,What is AI?
